# Does PRIME's prediction actually pick high-TEP moments?

The closed loop stimulates when its classifier says the current brain state will produce a
large TEP. This notebook asks whether that is true, and is careful to separate two claims that
are easy to conflate:

| | question | what it needs |
|---|---|---|
| **Q1 · discrimination** | Among delivered pulses, does a higher prediction go with a larger TEP? | correlation between `prediction_probability` and TEP |
| **Q2 · effect** | Do PRIME-chosen moments actually produce larger TEPs than clock-chosen ones? | PRIME singles vs `predetermined_single` controls |

Q1 is the mechanism, Q2 is the intervention. **Q1 can look weak while Q2 is real, and Q2 can
look real for reasons that have nothing to do with brain state.** Both traps are live here, and
both are quantified rather than assumed away:

- **Q1 is range-restricted.** Every delivered PRIME pulse had `p >= 0.5`, so only the top part
  of the predictor's range is ever paired with an outcome. That attenuates any correlation
  mechanically, regardless of how good the model is. Section 6 measures the attenuation.
- **Q2 is confounded by timing.** PRIME fires as soon as it can, at ~2.5 s. Predetermined
  trials wait a random 2.5–5.5 s. The two conditions therefore differ systematically in the
  interval since the previous pulse, before any brain state is considered. Section 3 measures
  it and section 5 adjusts for it.

There is also a ceiling: if TEP amplitude has no structure that persists for a few seconds,
nothing measurable at −65 ms can predict it. Section 2 estimates that ceiling from the data
before asking whether the model reaches it.

---

| § | contents |
|---|---|
| 0 | Setup and the analysis table |
| 1 | What is in the sample, and what was lost |
| 2 | The predictability ceiling: does TEP amplitude have any temporal structure? |
| 3 | Confound audit: everything that differs between the conditions besides brain state |
| 4 | **Q1** — does the prediction discriminate TEP amplitude? |
| 5 | **Q2** — PRIME versus predetermined controls |
| 6 | How much the trigger threshold hides |
| 7 | The experiment that would settle it |
| 8 | Verdict |


---
## 0 · Setup

Same path resolution as the diagnostics notebook: sessions come from the curated **BIDS** tree,
never from `Import`. If `diagnostics/per_trial_diagnostics.csv` exists (written by
`PRIME_online_diagnostics.ipynb`) it is used for the replayed amplitudes and gate outcomes;
otherwise the notebook falls back to the online logs alone.


In [ ]:
from __future__ import annotations

import sys, os, json, warnings, platform
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({"figure.dpi": 110, "figure.figsize": (11, 3.6),
                            "axes.grid": True, "grid.alpha": 0.25,
                            "axes.spines.top": False, "axes.spines.right": False})

# ---------------------------------------------------------------- configuration
LOOP_ROOT_OVERRIDE   = None
SUBJECT_DIR_OVERRIDE = None
OUT_DIR_OVERRIDE     = None
LOOP_ROOT_CANDIDATES = [r"D:\Linus\Loop"]
SESSION_GLOB = "BIDS/sub-*/ses-*/*/prime"
LOOP_MARKERS = ("prime", "BIDS")
STALE = ("old", "backup", "bak", "archive", "copy", "_prev")

RNG = np.random.default_rng(20260807)     # every resampling result below is reproducible
N_BOOT = 10000
N_PERM = 10000


def _is_stale(p: Path) -> bool:
    return any(s in part.lower() for part in p.parts for s in STALE)


def _rank(p: Path):
    return (_is_stale(p), len(p.parts), -p.stat().st_mtime)


def _looks_like_loop(p: Path) -> bool:
    return p.is_dir() and all((p / m).is_dir() for m in LOOP_MARKERS)


def _find_loop_root():
    if LOOP_ROOT_OVERRIDE:
        return Path(LOOP_ROOT_OVERRIDE)
    if os.environ.get("LOOP_ROOT"):
        return Path(os.environ["LOOP_ROOT"])
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if _looks_like_loop(d):
            return d
    for c in LOOP_ROOT_CANDIDATES:
        if _looks_like_loop(Path(c)):
            return Path(c)
    return None


LOOP_ROOT = _find_loop_root()
if LOOP_ROOT is None:
    raise FileNotFoundError(
        "Could not find the Loop root. Set LOOP_ROOT_OVERRIDE, or run from inside the folder "
        "that contains both `prime` and `BIDS`.")

if SUBJECT_DIR_OVERRIDE:
    SUBJECT_DIR = Path(SUBJECT_DIR_OVERRIDE)
else:
    sess = sorted((p for p in LOOP_ROOT.glob(SESSION_GLOB)
                   if (p / "trials_intervention.csv").exists()), key=_rank)
    if not sess:
        raise FileNotFoundError(f"No session with trials_intervention.csv under "
                                f"{LOOP_ROOT / SESSION_GLOB}. Set SUBJECT_DIR_OVERRIDE.")
    SUBJECT_DIR = sess[0]
    if len(sess) > 1:
        print(f"{len(sess)} sessions found; set SUBJECT_DIR_OVERRIDE to pick another:")
        for s in sess:
            print(f"    {'-> ' if s == SUBJECT_DIR else '   '}{s}")

SES_DIR = next((q for q in SUBJECT_DIR.parents if q.name.startswith("ses-")), SUBJECT_DIR)
SUBJECT_ID = next((q.name for q in SUBJECT_DIR.parents if q.name.startswith("sub-")), "?")
SESSION_ID = SES_DIR.name if SES_DIR.name.startswith("ses-") else "?"
OUT_DIR = Path(OUT_DIR_OVERRIDE) if OUT_DIR_OVERRIDE else SES_DIR / "diagnostics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"session : {SUBJECT_ID} / {SESSION_ID}")
print(f"data    : {SUBJECT_DIR}")
print(f"outputs : {OUT_DIR}")


In [ ]:
# ---------------------------------------------------------------- load
TRIALS = pd.read_csv(SUBJECT_DIR / "trials_intervention.csv")
PRED_LOG = (pd.read_csv(SUBJECT_DIR / "prime_predictions.csv")
            if (SUBJECT_DIR / "prime_predictions.csv").exists() else None)

DIAG_PATH = OUT_DIR / "per_trial_diagnostics.csv"
DIAG = pd.read_csv(DIAG_PATH) if DIAG_PATH.exists() else None
print(f"trials_intervention.csv : {TRIALS.shape}")
print(f"prime_predictions.csv   : {None if PRED_LOG is None else PRED_LOG.shape}")
print(f"per_trial_diagnostics   : {None if DIAG is None else DIAG.shape}"
      + ("" if DIAG is not None else "   (run PRIME_online_diagnostics.ipynb to add it)"))

IV = TRIALS[TRIALS.stage.str.startswith("intervention")].copy()
IV = IV.sort_values("pulse_time").reset_index(drop=True)

# --- derived trial-level variables ----------------------------------------------------
IV["isi"] = IV.pulse_time.diff()                       # seconds since the previous pulse
IV["prev_condition"] = IV.condition.shift()
IV["is_prime"] = IV.condition == "prime_single_pulse"
IV["is_ctrl"] = IV.condition == "predetermined_single"
IV["is_single"] = IV.is_prime | IV.is_ctrl
IV["trial_order"] = np.arange(len(IV))
IV["t_min"] = (IV.pulse_time - IV.pulse_time.min()) / 60
IV["mini_block"] = IV.groupby("stage").cumcount() // 20
IV["kept"] = ~IV.postprocessing_failed.astype(bool)

if DIAG is not None:
    IV = IV.merge(DIAG[["stage", "trial", "outcome", "amp_free", "r2_free", "latency_ms",
                        "ocular_z", "global_z", "local_z_max"]]
                  .rename(columns={"trial": "trial_in_stage", "amp_free": "amp_replay"}),
                  on=["stage", "trial_in_stage"], how="left")

print(f"\nintervention trials: {len(IV)}")
print(IV.condition.value_counts().to_frame("n").to_string())


### 0.1 · Choosing the outcome variable

Two candidates, and they answer different questions.

- **`tep_amplitude_raw`** — the dipole amplitude in physical units. This is what "a larger TEP"
  literally means, but it drifts over the session, so a raw comparison mixes brain state with
  slow drift.
- **`tep_amplitude`** (the label) — the same amplitude after EWMA detrending, z-scoring against
  calibration, and an ECDF transform to `[0, 1]`. **This is what the classifier was trained to
  predict**, and it is drift-robust by construction.

For **Q1** the label is the correct target: asking whether the model predicts its own training
target is the fair test. For **Q2** both are reported — the label answers "does PRIME
preferentially land on locally-high trials", the raw answers "are the resulting TEPs bigger".

One caveat carried throughout: the label is a *relative* quantity computed against a running
history that both conditions contribute to, so the two conditions are not statistically
independent through the normalizer. That weakens nothing in Q1 but is worth remembering in Q2.


In [ ]:
TARGETS = {}
if "tep_amplitude" in IV.columns and IV.tep_amplitude.notna().any():
    TARGETS["label"] = "tep_amplitude"
if "tep_amplitude_raw" in IV.columns and IV.tep_amplitude_raw.notna().any():
    TARGETS["raw"] = "tep_amplitude_raw"
if not TARGETS:
    raise RuntimeError("No TEP outcome column found in trials_intervention.csv.")

PRIMARY = "label" if "label" in TARGETS else "raw"
print("outcome variables available:")
for k, v in TARGETS.items():
    s = IV.loc[IV.is_single, v].dropna()
    print(f"  {k:6s} -> {v:20s} n={len(s):4d}  "
          f"median {s.median():.4g}  IQR [{s.quantile(.25):.4g}, {s.quantile(.75):.4g}]")
print(f"\nprimary target for Q1: {TARGETS[PRIMARY]}  ({PRIMARY})")


---
## 1 · What is in the sample

Only single-pulse trials can be used. Triplets are excluded because their "TEP amplitude" is
dominated by the second and third pulses landing inside the 38–50 ms dipole window — it is not
a TEP, and it was deliberately withheld from the normalizer for that reason.


In [ ]:
S = IV[IV.is_single].copy()
print(f"single-pulse trials delivered : {len(S)}")
print(f"  prime_single_pulse          : {int(S.is_prime.sum())}")
print(f"  predetermined_single        : {int(S.is_ctrl.sum())}")

A = S[S.kept & S[TARGETS[PRIMARY]].notna()].copy()
print(f"\nanalysable (survived post-processing, TEP present): {len(A)}")
print(A.condition.value_counts().to_frame("n").to_string())

lost = S[~S.kept]
print(f"\nlost to post-processing rejection: {len(lost)}")
print((lost.condition.value_counts() / S.condition.value_counts() * 100)
      .round(1).to_frame("% of delivered").dropna().to_string())
print("\nThe two conditions do NOT lose trials at the same rate. Section 3.3 tests whether")
print("that biases the comparison, and section 5 runs a sensitivity analysis for it.")

# Statistical power available, given the sample actually in hand.
n1, n2 = int(A.is_prime.sum()), int(A.is_ctrl.sum())
if n1 and n2:
    for d in (0.2, 0.35, 0.5, 0.8):
        se = np.sqrt(1 / n1 + 1 / n2)
        ncp = d / se
        crit = stats.norm.ppf(0.975)
        power = stats.norm.sf(crit - ncp) + stats.norm.cdf(-crit - ncp)
        print(f"  power to detect Cohen's d = {d:.2f} at alpha .05 (two-sided): {power:.2f}")
    d_det = (stats.norm.ppf(0.975) + stats.norm.ppf(0.80)) * np.sqrt(1 / n1 + 1 / n2)
    print(f"\nsmallest effect detectable with 80% power: d = {d_det:.2f}")
    print("Any 'no significant difference' below has to be read against that number.")


---
## 2 · The predictability ceiling

Before asking whether the model predicts TEP amplitude, ask whether TEP amplitude is
predictable **at all** on this timescale.

The classifier sees a 50 ms window ending 15 ms before the pulse. For that to carry information
about the response, TEP amplitude must share variance with something that persists over at
least a few seconds. If consecutive single-pulse trials are statistically independent, then the
predictable fraction of variance is near zero and no classifier — however good — can do much.

Two probes:

1. **Lag-1 autocorrelation** of the trial sequence. Detrended, so it measures short-range
   structure rather than slow drift.
2. **Split-half reliability by time bin**: if amplitude were pure noise, adjacent bins would be
   uncorrelated.


In [ ]:
seq = A.sort_values("pulse_time").reset_index(drop=True)
y_lab = seq[TARGETS[PRIMARY]].to_numpy(float)

print(f"autocorrelation of the trial-ordered outcome ({TARGETS[PRIMARY]}, n={len(y_lab)})")
print(f"{'lag':>4} {'r':>8} {'p':>9}")
ac = []
for lag in range(1, 11):
    a, b = y_lab[:-lag], y_lab[lag:]
    r, p = stats.pearsonr(a, b)
    ac.append((lag, r, p))
    print(f"{lag:>4} {r:>+8.3f} {p:>9.3f}")
AC = pd.DataFrame(ac, columns=["lag", "r", "p"])

r1, p1 = AC.loc[0, "r"], AC.loc[0, "p"]
ci = np.tanh(np.arctanh(r1) + np.array([-1, 1]) * 1.96 / np.sqrt(len(y_lab) - 3))
print(f"\nlag-1: r = {r1:+.3f}  95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}]  p = {p1:.3f}")
print(f"shared variance between consecutive trials: {100*r1**2:.1f}%")

if abs(r1) < 0.1 or p1 > 0.05:
    print("\n  *** The outcome behaves like an independent draw trial to trial. ***")
    print("  With no serial structure there is very little for a pre-stimulus predictor to")
    print("  latch onto, and any true effect must be small. This bounds everything below:")
    print("  a weak Q1 result would be expected even from a perfectly trained model.")
else:
    print(f"\n  There is measurable serial structure, so a pre-stimulus predictor has")
    print(f"  something to work with. An upper bound on what any predictor of the immediately")
    print(f"  preceding state could explain is roughly r^2 = {100*r1**2:.1f}% of the variance.")


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
ax[0].stem(AC.lag, AC.r)
ci95 = 1.96 / np.sqrt(len(y_lab))
ax[0].axhspan(-ci95, ci95, color="crimson", alpha=.15)
ax[0].axhline(0, color="k", lw=.8)
ax[0].set(xlabel="lag (trials)", ylabel="autocorrelation",
          title="Serial structure in TEP amplitude\n(shaded = 95% null band)")

ax[1].plot(seq.t_min, y_lab, ".", ms=3, alpha=.45)
ax[1].plot(seq.t_min, pd.Series(y_lab).rolling(25, min_periods=5, center=True).mean(),
           lw=1.6, color="crimson")
ax[1].set(xlabel="session time (min)", ylabel=TARGETS[PRIMARY],
          title="Outcome over the session (rolling mean, 25)")

ax[2].scatter(y_lab[:-1], y_lab[1:], s=7, alpha=.4)
ax[2].set(xlabel="trial n", ylabel="trial n+1", title=f"Lag-1 scatter (r = {r1:+.3f})")
plt.tight_layout(); plt.show()

if "raw" in TARGETS and PRIMARY != "raw":
    yr = seq[TARGETS["raw"]].to_numpy(float)
    rr, pr = stats.pearsonr(yr[:-1], yr[1:])
    print(f"same check on the RAW amplitude: lag-1 r = {rr:+.3f} (p = {pr:.3f})")
    print("The raw series also carries slow drift, so its autocorrelation is the more")
    print("optimistic of the two. The label's value is the honest short-range number.")


---
## 3 · Confound audit

Everything that differs between PRIME trials and predetermined controls **other than the brain
state at the moment of stimulation**. Each of these can manufacture — or mask — a difference in
TEP amplitude on its own.


### 3.1 · Time since the previous pulse

PRIME fires at the first tick that clears QC and threshold, which is 2.5 s after the previous
pulse on the overwhelming majority of trials. Predetermined trials draw a uniform ITI in
[2.5, 5.5] s. So the two conditions systematically differ in how much time the cortex has had
to recover — before any brain state is considered.


In [ ]:
isi = S.dropna(subset=["isi"])
print("interval since the previous pulse (s):")
display(isi.groupby("condition").isi.describe()[["count", "mean", "50%", "std", "min", "max"]].round(3))

a_isi = isi.loc[isi.is_prime, "isi"]
b_isi = isi.loc[isi.is_ctrl, "isi"]
u = stats.mannwhitneyu(a_isi, b_isi)
print(f"\nPRIME median {a_isi.median():.3f} s   control median {b_isi.median():.3f} s"
      f"   difference {b_isi.median() - a_isi.median():+.3f} s")
print(f"Mann-Whitney U p = {u.pvalue:.3g}")

ISI_CONFOUNDED = u.pvalue < 0.05
if ISI_CONFOUNDED:
    print("\n  *** The conditions differ in preceding interval. Any raw PRIME-vs-control")
    print("      comparison is contaminated by it, and section 5 must adjust. ***")

# Does the interval actually move the outcome? If not, the confound is harmless.
for name, col in TARGETS.items():
    d = A.dropna(subset=["isi", col])
    r, p = stats.spearmanr(d.isi, d[col])
    print(f"\nISI vs {col}: spearman rho = {r:+.3f} (p = {p:.3f}, n = {len(d)})")
    if p < 0.05:
        print("   The interval does predict the outcome, so the confound is ACTIVE.")
    else:
        print("   No detectable relationship, so the imbalance is unlikely to matter much")
        print("   -- but section 5 still reports the adjusted estimate.")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
for cond, colr in [("prime_single_pulse", "#5b8ff9"), ("predetermined_single", "#5ad8a6")]:
    v = isi.loc[isi.condition == cond, "isi"]
    ax[0].hist(v, bins=np.linspace(2, 8, 60), histtype="step", lw=1.5, color=colr,
               label=f"{cond} (median {v.median():.2f} s)", density=True)
ax[0].set(xlabel="interval since previous pulse (s)", ylabel="density",
          title="The two conditions do not sample the same intervals")
ax[0].legend(fontsize=7)

d = A.dropna(subset=["isi", TARGETS[PRIMARY]])
d = d[d.isi < d.isi.quantile(.99)]
ax[1].scatter(d.isi, d[TARGETS[PRIMARY]], s=8, alpha=.4,
              c=["#5b8ff9" if x else "#5ad8a6" for x in d.is_prime])
if len(d) > 10:
    z = np.polyfit(d.isi, d[TARGETS[PRIMARY]], 1)
    xs = np.linspace(d.isi.min(), d.isi.max(), 50)
    ax[1].plot(xs, np.polyval(z, xs), "r--", lw=1.2)
ax[1].set(xlabel="interval since previous pulse (s)", ylabel=TARGETS[PRIMARY],
          title="Does the interval move the outcome?")
plt.tight_layout(); plt.show()


### 3.2 · What preceded each trial

A triplet delivers three pulses; a single delivers one. If one condition is more often preceded
by triplets, it receives a different recent dose.


In [ ]:
prev = pd.crosstab(S.prev_condition, S.condition, normalize="columns") * 100
display(prev.round(1))
tab = pd.crosstab(S.prev_condition, S.condition)
chi = stats.chi2_contingency(tab.values)
print(f"chi-square on the preceding-condition mix: p = {chi.pvalue:.3f}")
print("  A high p means the two conditions are preceded by a similar mix, so recent dose is")
print("  balanced. A low p means it is not, and belongs in the adjustment model.")
PREV_CONFOUNDED = chi.pvalue < 0.05


### 3.3 · Differential loss to post-processing

If one condition loses more trials, and the lost trials are not a random subset, the surviving
samples are not comparable.


In [ ]:
tab = pd.crosstab(S.condition, S.kept)
tab.columns = ["rejected", "kept"]
tab["% rejected"] = (100 * tab.rejected / tab.sum(axis=1)).round(1)
display(tab)
ft = stats.fisher_exact(pd.crosstab(S.condition, S.kept).values)
print(f"Fisher exact p = {ft.pvalue:.3g}")
REJ_CONFOUNDED = ft.pvalue < 0.05
if REJ_CONFOUNDED:
    print("\n  *** Rejection is condition-dependent. PRIME only fires when the pre-stimulus QC")
    print("      has already passed, so its trials are pre-screened for clean data while the")
    print("      controls are not. Section 5.3 bounds how much this could distort Q2. ***")

if DIAG is not None and "outcome" in S.columns:
    print("\nwhich gate did the losing:")
    display(pd.crosstab(S.condition, S.outcome))


### 3.4 · Position in time

The two conditions are interleaved 6:2 inside every 20-trial mini-block, which controls drift
well. This confirms it rather than assuming it.


In [ ]:
print("mean session time (min) per condition:")
display(S.groupby("condition").t_min.agg(["mean", "std"]).round(2))
tt = stats.ttest_ind(S.loc[S.is_prime, "t_min"], S.loc[S.is_ctrl, "t_min"], equal_var=False)
print(f"Welch t-test on session time: p = {tt.pvalue:.3f}   "
      f"({'balanced' if tt.pvalue > .05 else 'IMBALANCED'})")

print("\ntrials per block:")
display(pd.crosstab(S.stage, S.condition))
print("\nposition within the mini-block (should be spread, not clustered):")
S2 = S.copy()
S2["pos_in_mini"] = S2.groupby(["stage", "mini_block"]).cumcount()
display(pd.crosstab(S2.condition, S2.pos_in_mini > 9).rename(
    columns={False: "first half", True: "second half"}))

CONFOUNDS = pd.DataFrame([
    ("interval since previous pulse", ISI_CONFOUNDED),
    ("preceding condition mix", PREV_CONFOUNDED),
    ("differential post-processing rejection", REJ_CONFOUNDED),
    ("position in session time", tt.pvalue < 0.05),
], columns=["confound", "imbalanced"])
print("\nsummary:")
display(CONFOUNDS)


---
## 4 · Q1 — does the prediction discriminate TEP amplitude?

Among the PRIME single-pulse trials that were delivered and survived preprocessing, does a
higher `prediction_probability` go with a larger TEP?

**Read this section as a lower bound.** Only trials with `p >= 0.5` were ever delivered, so the
predictor's range is truncated at the median of its own decision axis. That attenuates any
correlation for purely mechanical reasons. Section 6 quantifies by how much.

One thing that is *not* a problem: the model is finetuned on each single-pulse trial right
after it happens, so at trial *n* it has only seen trials 1…*n*−1. Every prediction below is a
genuine forward prediction, never an in-sample fit.


In [ ]:
# ---------------------------------------------------------------- statistical helpers
def boot_ci(fn, *arrays, n=N_BOOT, alpha=0.05, paired=True):
    """Percentile bootstrap CI for any statistic of one or more equal-length arrays."""
    arrays = [np.asarray(a) for a in arrays]
    k = len(arrays[0])
    out = np.empty(n)
    for i in range(n):
        idx = RNG.integers(0, k, k)
        out[i] = fn(*[a[idx] for a in arrays])
    return float(np.nanpercentile(out, 100 * alpha / 2)), float(np.nanpercentile(out, 100 * (1 - alpha / 2)))


def perm_p(fn, x, y, n=N_PERM, tail="two"):
    """Permutation p-value for a statistic of paired vectors, shuffling y."""
    obs = fn(x, y)
    y = np.asarray(y).copy()
    cnt = 0
    for _ in range(n):
        RNG.shuffle(y)
        s = fn(x, y)
        if tail == "two":
            cnt += abs(s) >= abs(obs)
        elif tail == "greater":
            cnt += s >= obs
        else:
            cnt += s <= obs
    return obs, (cnt + 1) / (n + 1)


def auc(x, labels):
    """Probability that a positive case outranks a negative one."""
    x, labels = np.asarray(x), np.asarray(labels).astype(bool)
    n1, n0 = labels.sum(), (~labels).sum()
    if n1 == 0 or n0 == 0:
        return np.nan
    r = stats.rankdata(x)
    return (r[labels].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def ols(y, X, names):
    """Least squares with classical standard errors. statsmodels is not required."""
    y = np.asarray(y, float)
    X = np.column_stack([np.ones(len(y))] + [np.asarray(c, float) for c in X])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    dof = len(y) - X.shape[1]
    s2 = resid @ resid / dof
    se = np.sqrt(np.diag(s2 * np.linalg.pinv(X.T @ X)))
    t = beta / se
    return pd.DataFrame({"term": ["intercept"] + list(names), "beta": beta, "se": se,
                         "t": t, "p": 2 * stats.t.sf(np.abs(t), dof)}).round(4), resid


def hedges_g(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    d = (a.mean() - b.mean()) / sp
    return d * (1 - 3 / (4 * (na + nb) - 9))          # small-sample correction


print("helpers ready: boot_ci, perm_p, auc, ols, hedges_g")
print(f"bootstrap resamples {N_BOOT}, permutations {N_PERM}, seed fixed")


In [ ]:
# ---------------------------------------------------------------- the Q1 sample
Q1 = A[A.is_prime & A.prediction_probability.notna()].copy()
print(f"PRIME single-pulse trials with both a prediction and a TEP: {len(Q1)}")
print(f"prediction range actually observed: [{Q1.prediction_probability.min():.3f}, "
      f"{Q1.prediction_probability.max():.3f}]   sd {Q1.prediction_probability.std():.3f}")
print(f"  (the trigger threshold is 0.5, so nothing below it can appear here)")

rows = []
for name, col in TARGETS.items():
    d = Q1.dropna(subset=[col])
    x, y = d.prediction_probability.to_numpy(float), d[col].to_numpy(float)
    pr, pp = stats.pearsonr(x, y)
    sr, sp_ = stats.spearmanr(x, y)
    lo, hi = boot_ci(lambda a, b: stats.pearsonr(a, b)[0], x, y)
    _, perm = perm_p(lambda a, b: stats.pearsonr(a, b)[0], x, y)
    rows.append(dict(target=col, n=len(d), pearson_r=pr, ci_lo=lo, ci_hi=hi,
                     p_parametric=pp, p_permutation=perm, spearman_rho=sr, p_spearman=sp_))
Q1CORR = pd.DataFrame(rows).round(4)
display(Q1CORR)

print("The bootstrap CI and the permutation p are the ones to trust: they make no normality")
print("assumption, and the permutation null is exact for the sharp hypothesis of no")
print("association. If the CI spans zero, the data do not establish discrimination.")


In [ ]:
# ---------------------------------------------------------------- binned view
col = TARGETS[PRIMARY]
d = Q1.dropna(subset=[col]).copy()
nb = 5
d["pbin"] = pd.qcut(d.prediction_probability, nb, duplicates="drop")
# Note the explicit rename: "count"/"mean"/"sem"/"median" all collide with DataFrame methods,
# so attribute access would silently return the method rather than the column.
g = (d.groupby("pbin", observed=True)[col].agg(["count", "mean", "sem", "median"])
       .rename(columns={"count": "n", "mean": "m", "sem": "se", "median": "med"}))
g["p_mid"] = [iv.mid for iv in g.index]
display(g.round(4))

fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
ax[0].errorbar(g["p_mid"], g["m"], yerr=g["se"], fmt="o-", capsize=3)
ax[0].set(xlabel="prediction probability (bin centre)", ylabel=f"mean {col}",
          title=f"Outcome by prediction quantile ({nb} bins)")

ax[1].scatter(d.prediction_probability, d[col], s=10, alpha=.45)
z = np.polyfit(d.prediction_probability, d[col], 1)
xs = np.linspace(d.prediction_probability.min(), d.prediction_probability.max(), 50)
ax[1].plot(xs, np.polyval(z, xs), "r--", lw=1.3)
ax[1].set(xlabel="prediction probability", ylabel=col,
          title=f"r = {stats.pearsonr(d.prediction_probability, d[col])[0]:+.3f}")

hi_lo = d[col] > d[col].median()
Q1_AUC = auc(d.prediction_probability, hi_lo)
Q1_AUC_LO, Q1_AUC_HI = boot_ci(lambda p_, l_: auc(p_, l_ > np.median(l_)),
                               d.prediction_probability.to_numpy(), d[col].to_numpy())
srt = np.argsort(-d.prediction_probability.to_numpy())
lab = hi_lo.to_numpy()[srt]
tpr = np.concatenate([[0], np.cumsum(lab) / max(lab.sum(), 1)])
fpr = np.concatenate([[0], np.cumsum(~lab) / max((~lab).sum(), 1)])
ax[2].plot(fpr, tpr, lw=1.5)
ax[2].plot([0, 1], [0, 1], "k--", lw=.8)
ax[2].set(xlabel="false positive rate", ylabel="true positive rate",
          title=f"Discriminating above-median TEP\n"
                f"AUC = {Q1_AUC:.3f}  95% CI [{Q1_AUC_LO:.3f}, {Q1_AUC_HI:.3f}]")
plt.tight_layout(); plt.show()

print(f"AUC = {Q1_AUC:.3f}, 95% CI [{Q1_AUC_LO:.3f}, {Q1_AUC_HI:.3f}]")
print("AUC 0.5 = coin flip. The CI including 0.5 means the prediction cannot be shown to")
print("separate above- from below-median trials within the delivered range.")


In [ ]:
# ---------------------------------------------------------------- stability across blocks
rows = []
for blk, gg in Q1.dropna(subset=[col]).groupby("stage"):
    if len(gg) < 10:
        continue
    r, p = stats.pearsonr(gg.prediction_probability, gg[col])
    rows.append(dict(block=blk, n=len(gg), pearson_r=r, p=p))
BLKCORR = pd.DataFrame(rows).round(4)
display(BLKCORR)

if len(BLKCORR) > 1:
    zs = np.arctanh(BLKCORR.pearson_r.clip(-.999, .999))
    w = BLKCORR.n - 3
    zbar = (zs * w).sum() / w.sum()
    Qstat = (w * (zs - zbar) ** 2).sum()
    het_p = stats.chi2.sf(Qstat, len(zs) - 1)
    print(f"pooled r across blocks (Fisher z): {np.tanh(zbar):+.3f}")
    print(f"heterogeneity across blocks: Q = {Qstat:.2f}, p = {het_p:.3f}"
          f"   ({'consistent' if het_p > .05 else 'blocks disagree'})")
    print("A model that genuinely works should hold up in every block, not just one.")


### 4.1 · Negative controls

If the prediction tracks brain state, it should relate to the outcome and *not* to things it
has no business predicting. But the candidate variables split into two classes that must not be
read the same way.

**Decision-rule-linked — a correlation here is an artifact, not a finding.** `prime_attempts`,
`isi` and `qc_failures` are all downstream of the stopping rule. The loop fires on the *first*
tick that clears 0.5, so a trial whose predictions run high triggers immediately with a high
value, while a trial whose predictions run low waits and eventually crosses with a value barely
over the threshold. And for a PRIME trial `isi` is essentially `2.5 s + 10 ms × (attempts +
qc_failures)` — the same quantity in different units. Correlations with these are guaranteed by
the mechanism and say nothing about what the model is reading.

**Genuine nuisance — a correlation here IS a red flag.** Data-quality measures and session time
are not part of the decision rule. If the prediction tracks these more strongly than it tracks
the TEP, the model is keying on artifact or drift rather than on physiology.

Only the second class is compared against the target.


In [ ]:
RULE_LINKED = ["prime_attempts", "isi", "qc_failures"]
NUISANCE = ["ocular_z", "global_z", "local_z_max", "r2_free", "t_min"]

rows = []
for c in [x for x in RULE_LINKED + NUISANCE if x in Q1.columns]:
    dd = Q1.dropna(subset=[c, "prediction_probability"])
    if len(dd) < 15 or dd[c].nunique() < 3:
        continue
    r, p = stats.spearmanr(dd.prediction_probability, dd[c])
    rows.append(dict(variable=c, kind="decision-rule" if c in RULE_LINKED else "nuisance",
                     n=len(dd), spearman_rho=r, p=p))
NEG = pd.DataFrame(rows).round(4).sort_values(["kind", "p"])
display(NEG)

d0 = Q1.dropna(subset=[col])
r_target = abs(stats.spearmanr(d0.prediction_probability, d0[col]).statistic)
print(f"|rho| with the intended target ({col}): {r_target:.3f}\n")

# Make the mechanical link visible rather than asserting it.
if "prime_attempts" in Q1.columns:
    att = (Q1.groupby(Q1.prime_attempts.clip(upper=5))
             .prediction_probability.agg(["count", "mean", "median"]).round(3))
    att.index.name = "attempts (5 = 5 or more)"
    print("probability at the moment of triggering, by how many attempts it took:")
    display(att)
    print("Fewer attempts -> higher triggering probability. That is the stopping rule showing")
    print("through, not the model reading the clock.\n")

NUIS = NEG[NEG.kind == "nuisance"]
if len(NUIS):
    worst = NUIS.iloc[0]
    print(f"strongest genuine-nuisance association: {worst.variable} "
          f"|rho| = {abs(worst.spearman_rho):.3f} (p = {worst.p:.4f})")
    NEG_FLAG = abs(worst.spearman_rho) > r_target and worst.p < 0.05
    if NEG_FLAG:
        print(f"\n  *** The prediction tracks {worst.variable} more strongly than it tracks the")
        print(f"      TEP ({abs(worst.spearman_rho):.3f} vs {r_target:.3f}). Worth explaining")
        print("      before crediting the model with reading brain state. ***")
        sig_nuis = NUIS[(NUIS.p < 0.05)]
        if len(sig_nuis):
            print(f"      significant nuisance associations: "
                  f"{', '.join(sig_nuis.variable)}")
    else:
        print("  No genuine nuisance variable outranks the intended target.")
else:
    NEG_FLAG = False


---
## 5 · Q2 — PRIME versus predetermined controls

The intervention question. `predetermined_single` trials are single pulses delivered on a
clock, ignoring brain state, interleaved 2-per-20 with the PRIME trials. They are the correct
comparison.

Three passes: unadjusted, adjusted for the confounds section 3 identified, and a bounds
analysis for the trials that were lost unequally between conditions.


In [ ]:
# ---------------------------------------------------------------- 5.1 unadjusted
rows = []
for name, tcol in TARGETS.items():
    d = A.dropna(subset=[tcol])
    a_ = d.loc[d.is_prime, tcol].to_numpy(float)
    b_ = d.loc[d.is_ctrl, tcol].to_numpy(float)
    if len(a_) < 5 or len(b_) < 5:
        continue
    w = stats.ttest_ind(a_, b_, equal_var=False)
    mw = stats.mannwhitneyu(a_, b_)
    g_ = hedges_g(a_, b_)
    # Unpaired bootstrap: resample each group independently, since the groups are separate.
    diffs = np.empty(N_BOOT)
    for i in range(N_BOOT):
        diffs[i] = (RNG.choice(a_, len(a_), replace=True).mean()
                    - RNG.choice(b_, len(b_), replace=True).mean())
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    rows.append(dict(target=tcol, n_prime=len(a_), n_ctrl=len(b_),
                     mean_prime=a_.mean(), mean_ctrl=b_.mean(),
                     diff=a_.mean() - b_.mean(), ci_lo=lo, ci_hi=hi,
                     hedges_g=g_, p_welch=w.pvalue, p_mannwhitney=mw.pvalue,
                     auc=auc(np.r_[a_, b_], np.r_[np.ones(len(a_)), np.zeros(len(b_))])))
Q2 = pd.DataFrame(rows).round(4)
display(Q2)

print("`auc` here is the probability that a randomly chosen PRIME trial exceeds a randomly")
print("chosen control trial. 0.50 = no difference; 0.56 is a small effect, 0.64 medium.")


In [ ]:
fig, ax = plt.subplots(1, len(TARGETS) + 1, figsize=(5 * (len(TARGETS) + 1), 3.6))
ax = np.atleast_1d(ax)
for a_i, (name, tcol) in zip(ax, TARGETS.items()):
    d = A.dropna(subset=[tcol])
    data = [d.loc[d.is_prime, tcol], d.loc[d.is_ctrl, tcol]]
    parts = a_i.violinplot(data, showmedians=True, widths=.8)
    for pc, colr in zip(parts["bodies"], ["#5b8ff9", "#5ad8a6"]):
        pc.set_facecolor(colr); pc.set_alpha(.55)
    for i, dd in enumerate(data, start=1):
        a_i.scatter(np.full(len(dd), i) + RNG.normal(0, .04, len(dd)), dd, s=5, alpha=.35,
                    color="k", zorder=3)
    a_i.set_xticks([1, 2]); a_i.set_xticklabels(["PRIME", "control"])
    a_i.set(ylabel=tcol, title=f"{tcol}\nn = {len(data[0])} vs {len(data[1])}")

d = A.dropna(subset=[TARGETS[PRIMARY]])
a_ = d.loc[d.is_prime, TARGETS[PRIMARY]].to_numpy(float)
b_ = d.loc[d.is_ctrl, TARGETS[PRIMARY]].to_numpy(float)
diffs = np.array([RNG.choice(a_, len(a_), True).mean() - RNG.choice(b_, len(b_), True).mean()
                  for _ in range(N_BOOT)])
ax[-1].hist(diffs, bins=60, color="#888")
ax[-1].axvline(0, color="crimson", lw=1.5)
ax[-1].axvline(diffs.mean(), color="k", lw=1.2, ls="--")
ax[-1].set(xlabel="PRIME - control (bootstrap)", title="Bootstrap of the mean difference")
plt.tight_layout(); plt.show()
print(f"bootstrap: {100*(diffs > 0).mean():.1f}% of resamples favour PRIME "
      f"(50% would be pure chance)")


### 5.2 · Adjusted for the confounds

Least squares on the outcome with condition plus every imbalanced nuisance variable from
section 3. The coefficient on `is_prime` is the PRIME effect holding the others fixed.


In [ ]:
d = A.dropna(subset=[TARGETS[PRIMARY], "isi"]).copy()
d["prev_triplet"] = (d.prev_condition == "prime_triplet").astype(float)

specs = [
    ("condition only", ["is_prime"]),
    ("+ interval", ["is_prime", "isi"]),
    ("+ interval + preceding triplet", ["is_prime", "isi", "prev_triplet"]),
    ("+ interval + preceding triplet + time", ["is_prime", "isi", "prev_triplet", "t_min"]),
]
rows = []
for label, terms in specs:
    tab, _ = ols(d[TARGETS[PRIMARY]], [d[t].astype(float) for t in terms], terms)
    r = tab[tab.term == "is_prime"].iloc[0]
    rows.append(dict(model=label, beta_is_prime=r.beta, se=r.se, t=r.t, p=r.p, n=len(d)))
ADJ = pd.DataFrame(rows).round(4)
display(ADJ)

print("full model:")
tab, resid = ols(d[TARGETS[PRIMARY]],
                 [d[t].astype(float) for t in ["is_prime", "isi", "prev_triplet", "t_min"]],
                 ["is_prime", "isi", "prev_triplet", "t_min"])
display(tab)

if ADJ.beta_is_prime.abs().max() > 0 and np.sign(ADJ.beta_is_prime).nunique() > 1:
    print("\n  The sign of the PRIME coefficient flips between specifications -- the estimate")
    print("  is not robust to how the confounds are handled.")
else:
    shrink = 1 - abs(ADJ.beta_is_prime.iloc[-1]) / max(abs(ADJ.beta_is_prime.iloc[0]), 1e-12)
    print(f"\n  Adjusting shrinks the PRIME coefficient by {100*shrink:.0f}%.")


### 5.3 · Bounds for the unequal trial loss

The two conditions lost different fractions of trials, and the lost outcomes are unobserved.
Because the label is bounded in `[0, 1]`, exact worst-case and best-case bounds can be computed
without assuming anything about *why* trials were lost (Manski bounds). If the bounds straddle
zero, the data alone cannot settle the direction of the effect.


In [ ]:
tcol = TARGETS["label"] if "label" in TARGETS else TARGETS[PRIMARY]
bounded = tcol == TARGETS.get("label")
d_all = S.copy()
obs = d_all[d_all.kept & d_all[tcol].notna()]

nP, nC = int(d_all.is_prime.sum()), int(d_all.is_ctrl.sum())
oP = obs[obs.is_prime][tcol].to_numpy(float)
oC = obs[obs.is_ctrl][tcol].to_numpy(float)
mP, mC = nP - len(oP), nC - len(oC)
print(f"PRIME   : {len(oP)} observed, {mP} missing of {nP}")
print(f"control : {len(oC)} observed, {mC} missing of {nC}")

if bounded:
    lo_v, hi_v = 0.0, 1.0
    best = (oP.sum() + mP * hi_v) / nP - (oC.sum() + mC * lo_v) / nC
    worst = (oP.sum() + mP * lo_v) / nP - (oC.sum() + mC * hi_v) / nC
    naive = oP.mean() - oC.mean()
    print(f"\nobserved difference (complete cases): {naive:+.4f}")
    print(f"Manski bounds on the true difference : [{worst:+.4f}, {best:+.4f}]")
    print(f"bound width: {best - worst:.4f}")
    if worst < 0 < best:
        print("\n  The bounds straddle zero. With this much missingness the sign of the effect")
        print("  is not identified from the observed data alone -- it depends on assumptions")
        print("  about the trials that were lost.")
    else:
        print("\n  The bounds exclude zero, so the direction survives even the worst case.")

    # The bounds move BOTH sides at once. The tipping points below move one side at a time,
    # which is what shows where the fragility actually lives.
    print("\n  Tipping points -- what the unobserved trials would have to average for the")
    print("  difference to vanish, moving one condition at a time:")
    tips = []
    if mP > 0:
        need_P = (oC.mean() * nP - oP.sum()) / mP
        tips.append(("the missing PRIME trials", mP, need_P, oP.mean()))
    if mC > 0:
        need_C = (oP.mean() * nC - oC.sum()) / mC
        tips.append(("the missing control trials", mC, need_C, oC.mean()))
    for what, m_, need, obs_mean in tips:
        ok = 0 <= need <= 1
        note = ("ATTAINABLE, so the effect is fragile on this side" if ok
                else "OUTSIDE the label range, so this side cannot erase it alone")
        print(f"    {what} (n = {m_}) would have to average {need:+.3f}")
        print(f"      observed mean in that condition is {obs_mean:.3f}, label scale is 0-1")
        print(f"      -> {note}")

    print("\n  Reading the two together: the Manski bounds move both sides simultaneously, so")
    print("  they are wider than either tipping point. Whichever side has the larger share of")
    print("  missing trials is the one that drives the bound.")
    frac_P, frac_C = mP / nP, mC / nC
    print(f"    missing share -- PRIME {100*frac_P:.1f}%, control {100*frac_C:.1f}%"
          f"   ->  {'control' if frac_C > frac_P else 'PRIME'} side binds")
else:
    print("Outcome is unbounded; Manski bounds not applicable.")


### 5.4 · Per-block consistency


In [ ]:
rows = []
for blk, gg in A.dropna(subset=[TARGETS[PRIMARY]]).groupby("stage"):
    a_ = gg.loc[gg.is_prime, TARGETS[PRIMARY]]
    b_ = gg.loc[gg.is_ctrl, TARGETS[PRIMARY]]
    if len(a_) < 5 or len(b_) < 5:
        continue
    rows.append(dict(block=blk, n_prime=len(a_), n_ctrl=len(b_),
                     mean_prime=a_.mean(), mean_ctrl=b_.mean(),
                     diff=a_.mean() - b_.mean(),
                     p=stats.mannwhitneyu(a_, b_).pvalue))
BLK2 = pd.DataFrame(rows).round(4)
display(BLK2)
if len(BLK2):
    print(f"blocks favouring PRIME: {int((BLK2['diff'] > 0).sum())}/{len(BLK2)}")
    sgn = stats.binomtest(int((BLK2['diff'] > 0).sum()), len(BLK2), .5)
    print(f"sign test across blocks: p = {sgn.pvalue:.3f}")
    print("A real effect should point the same way in most blocks. All four in one direction")
    print("would be suggestive even if no single block reaches significance.")


---
## 6 · How much the trigger threshold hides

Q1's correlation is computed on a truncated sample: only `p >= 0.5` ever produced a pulse. This
is textbook **range restriction**, and it shrinks correlations for reasons that have nothing to
do with the model's quality.

The correction needs the predictor's spread in the *unrestricted* population. The prediction
log provides it: every attempt is recorded, including the ones that did not trigger. Those
non-triggering attempts are moments the model looked at and declined — exactly the missing part
of the range.

Thorndike's Case II:

$$r_{\text{unrestricted}} = \frac{r \cdot (S/s)}{\sqrt{1 + r^{2}\left(\frac{S^{2}}{s^{2}} - 1\right)}}$$

with $s$ the restricted SD and $S$ the unrestricted SD. It assumes linearity and equal residual
variance across the range — optimistic assumptions, so treat the result as an **upper bound**
on what the full range would have shown.


In [ ]:
if PRED_LOG is None:
    print("prime_predictions.csv is required for this section.")
else:
    allp = PRED_LOG.probability.dropna()
    trig = PRED_LOG.loc[PRED_LOG.triggered & ~PRED_LOG.forced, "probability"].dropna()
    delivered = Q1.prediction_probability.dropna()

    print(f"all attempts the model scored     : n = {len(allp):5d}  "
          f"sd = {allp.std():.4f}  range [{allp.min():.3f}, {allp.max():.3f}]")
    print(f"attempts that triggered a pulse   : n = {len(trig):5d}  sd = {trig.std():.4f}")
    print(f"delivered + analysable (Q1 sample): n = {len(delivered):5d}  sd = {delivered.std():.4f}")
    print(f"\nattempts declined by the threshold: {len(allp) - len(trig)} "
          f"({100*(1 - len(trig)/len(allp)):.1f}% of everything the model looked at)")

    S_un, s_re = allp.std(ddof=1), delivered.std(ddof=1)
    ratio = S_un / s_re
    print(f"\nSD ratio S/s = {ratio:.3f}")

    rows = []
    for _, r in Q1CORR.iterrows():
        rr = r.pearson_r
        corrected = (rr * ratio) / np.sqrt(1 + rr ** 2 * (ratio ** 2 - 1))
        rows.append(dict(target=r.target, r_observed=rr, r_corrected=corrected,
                         inflation=corrected / rr if rr else np.nan))
    RR = pd.DataFrame(rows).round(4)
    display(RR)

    print("Even the corrected value is an upper bound. If it is still small, the ceiling")
    print("in section 2 -- not the threshold -- is the binding constraint.")


In [ ]:
if PRED_LOG is not None:
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    bins = np.linspace(0, 1, 61)
    ax[0].hist(allp, bins=bins, color="#bbb", label=f"all attempts (n={len(allp)})")
    ax[0].hist(trig, bins=bins, color="#5b8ff9", alpha=.85,
               label=f"triggered (n={len(trig)})")
    ax[0].axvline(0.5, color="crimson", lw=1.5)
    ax[0].set(xlabel="prediction probability", ylabel="attempts",
              title="What the model saw vs what became a pulse")
    ax[0].legend(fontsize=7)

    # Illustrate the attenuation with the observed correlation structure.
    rr = Q1CORR.loc[Q1CORR.target == TARGETS[PRIMARY], "pearson_r"].iloc[0]
    fracs = np.linspace(0.2, 1.0, 40)
    sim = []
    xs_full = RNG.normal(size=4000)
    ys_full = rr * xs_full + np.sqrt(max(1 - rr ** 2, 1e-9)) * RNG.normal(size=4000)
    for f in fracs:
        thr = np.quantile(xs_full, 1 - f)
        m = xs_full >= thr
        sim.append(stats.pearsonr(xs_full[m], ys_full[m])[0] if m.sum() > 30 else np.nan)
    ax[1].plot(fracs * 100, sim, lw=1.5)
    ax[1].axhline(rr, color="crimson", lw=1, ls="--", label=f"observed r = {rr:+.3f}")
    kept_frac = 100 * len(trig) / len(allp)
    ax[1].axvline(kept_frac, color="k", lw=1, ls=":", label=f"{kept_frac:.0f}% kept")
    ax[1].set(xlabel="% of the range retained", ylabel="observable r",
              title=f"Attenuation from truncation\n(simulated at true r = {rr:+.3f})")
    ax[1].legend(fontsize=7)
    plt.tight_layout(); plt.show()
    print("The left panel is the concrete version of the problem: the model's own decision")
    print("threshold removes the lower part of its range before any outcome is measured.")


---
## 7 · The experiment that would settle it

Everything above is bounded by one design property: **outcomes exist only where the model
already said yes.** The fix does not require a new session — the data are already on disk.

**Retrospective rescoring.** `predetermined_single` trials fire on a clock, so their
pre-stimulus states are an *unselected* sample spanning the model's full range. Their
`*_pre_raw.npy` buffers were saved, and so were 801 classifier checkpoints. Scoring those
buffers with the checkpoint that was current at the time gives prediction-outcome pairs across
the whole range — the unrestricted sample Q1 is missing.

The cell below checks whether the ingredients are present and states the one real obstacle.


In [ ]:
ckpts = sorted(SUBJECT_DIR.glob("classifier_after_trial_*.pt"))
pre_bufs = sorted(SUBJECT_DIR.glob("intervention_block_*_pre_raw.npy"))
have_torch = False
try:
    import torch                      # noqa: F401
    have_torch = True
except Exception:
    pass

print(f"classifier checkpoints  : {len(ckpts)}")
print(f"pre-stimulus buffers    : {len(pre_bufs)}")
print(f"predetermined trials    : {int(S.is_ctrl.sum())}  <- the unselected sample")
print(f"torch available here    : {have_torch}")

print("\nWhat this makes possible, and what it does not:")
print("  POSSIBLE. Reload each checkpoint, run OnlinePredictor.predict on the saved")
print("  pre-stimulus buffer of every predetermined trial, and pair the result with that")
print("  trial's TEP. Predictions then span the full range instead of only p >= 0.5.")
print()
print("  OBSTACLE. save_checkpoint() stores only model.wrapped_model.state_dict() -- the")
print("  classifier weights. It does NOT store the TTA alignment state, which is an EMA")
print("  (beta = 0.99) over a rolling 50-trial covariance buffer and is updated inside")
print("  finetune() via adapt_alignment(). Loading a checkpoint alone therefore reproduces")
print("  the weights but not the whitening transform that was in force at the time.")
print()
print("  WORKAROUND. The alignment state is deterministic given the calibration covariances")
print("  and the ordered sequence of finetuned trials, both of which are recoverable. Replay")
print("  the session in order -- calibrate, then walk trial by trial calling finetune on the")
print("  single-pulse trials exactly as the online loop did -- and the alignment is rebuilt")
print("  exactly. Score the predetermined trials as you pass them.")
print()
print("  ONE LINE MAKES IT UNNECESSARY NEXT TIME. In process_pulse, for predetermined")
print("  trials, call predictor.predict(pre) and log it WITHOUT acting on it. That single")
print("  shadow prediction per control trial yields the unrestricted sample directly, with")
print("  no replay and no reconstruction.")


In [ ]:
# Sample size the rescoring design would deliver, and what it could resolve.
n_ctrl_ok = int((A.is_ctrl).sum())
if n_ctrl_ok > 10:
    for r_true in (0.10, 0.15, 0.20, 0.30):
        z = np.arctanh(r_true) * np.sqrt(n_ctrl_ok - 3)
        power = stats.norm.sf(1.96 - z) + stats.norm.cdf(-1.96 - z)
        print(f"  with n = {n_ctrl_ok} unrestricted trials, power to detect r = {r_true:.2f}: {power:.2f}")
    r_det = np.tanh((1.96 + 0.84) / np.sqrt(n_ctrl_ok - 3))
    print(f"\n  smallest correlation detectable at 80% power: r = {r_det:.3f}")
    print(f"  Pooling several sessions is the practical route to resolving anything smaller.")


---
## 8 · Verdict

Assembled from the numbers computed above, not written in advance. Re-running on another
session produces a different verdict.


In [ ]:
V = []


def verdict(topic, finding, strength):
    V.append(dict(topic=topic, finding=finding, strength=strength))


# --- ceiling
if abs(r1) < 0.1 or p1 > 0.05:
    verdict("ceiling",
            f"TEP amplitude shows no serial structure (lag-1 r = {r1:+.3f}, p = {p1:.3f}). "
            "Consecutive trials are effectively independent, so very little of the variance "
            "is available to ANY pre-stimulus predictor.", "binding constraint")
else:
    verdict("ceiling",
            f"TEP amplitude carries some serial structure (lag-1 r = {r1:+.3f}); up to about "
            f"{100*r1**2:.0f}% of variance is in principle predictable.", "context")

# --- Q1
q1 = Q1CORR[Q1CORR.target == TARGETS[PRIMARY]].iloc[0]
sig1 = (q1.ci_lo > 0) or (q1.ci_hi < 0)
verdict("Q1 discrimination",
        f"r = {q1.pearson_r:+.3f}, 95% CI [{q1.ci_lo:+.3f}, {q1.ci_hi:+.3f}], "
        f"permutation p = {q1.p_permutation:.3f} (n = {int(q1.n)}). "
        + ("The CI excludes zero." if sig1 else "The CI includes zero."),
        "supported" if sig1 else "not established")

auc_lo, auc_hi = Q1_AUC_LO, Q1_AUC_HI
verdict("Q1 discrimination (AUC)",
        f"AUC {Q1_AUC:.3f}, 95% CI [{auc_lo:.3f}, {auc_hi:.3f}] for separating above- from "
        "below-median trials.",
        "supported" if auc_lo > 0.5 else "not established")

# --- Q2
q2 = Q2[Q2.target == TARGETS[PRIMARY]].iloc[0]
sig2 = (q2.ci_lo > 0) or (q2.ci_hi < 0)
verdict("Q2 effect (unadjusted)",
        f"PRIME {q2.mean_prime:.3f} vs control {q2.mean_ctrl:.3f}, difference "
        f"{q2['diff']:+.3f} [{q2.ci_lo:+.3f}, {q2.ci_hi:+.3f}], Hedges g = {q2.hedges_g:+.3f}, "
        f"Mann-Whitney p = {q2.p_mannwhitney:.3f}.",
        "supported" if sig2 else "not established")

adj_last = ADJ.iloc[-1]
verdict("Q2 effect (adjusted)",
        f"After adjusting for interval, preceding condition and session time, the PRIME "
        f"coefficient is {adj_last.beta_is_prime:+.4f} (p = {adj_last.p:.3f}).",
        "supported" if adj_last.p < 0.05 else "not established")

# --- confounds
for _, r in CONFOUNDS.iterrows():
    if r.imbalanced:
        verdict("confound", f"{r.confound} differs between conditions.", "must be adjusted")

# --- negative controls
if NEG_FLAG:
    w = NEG[NEG.kind == "nuisance"].iloc[0]
    verdict("negative control",
            f"The prediction tracks {w.variable} (|rho| = {abs(w.spearman_rho):.3f}, "
            f"p = {w.p:.4f}) more strongly than it tracks the TEP (|rho| = {r_target:.3f}).",
            "red flag")

# --- power
verdict("power",
        f"With n = {n1} PRIME and n = {n2} control trials, the smallest effect detectable at "
        f"80% power is d = {d_det:.2f}. Effects smaller than that cannot be ruled out by this "
        "session.", "limitation")

# --- range restriction
if PRED_LOG is not None:
    rr_row = RR[RR.target == TARGETS[PRIMARY]].iloc[0]
    verdict("range restriction",
            f"{100*(1 - len(trig)/len(allp)):.0f}% of scored moments never became a pulse. "
            f"Correcting for truncation raises r from {rr_row.r_observed:+.3f} to at most "
            f"{rr_row.r_corrected:+.3f}.", "upper bound")

VERDICT = pd.DataFrame(V)
pd.set_option("display.max_colwidth", 150)
display(VERDICT)


In [ ]:
# ---------------------------------------------------------------- the one-paragraph answer
print("=" * 78)
print(f"ANSWER FOR {SUBJECT_ID} / {SESSION_ID}")
print("=" * 78)

established = sig1 or (auc_lo > 0.5)
effect = sig2 or (adj_last.p < 0.05)

if established and effect:
    tone = ("PRIME's prediction discriminates TEP amplitude AND the selected trials produce "
            "larger responses than clock-triggered controls.")
elif established and not effect:
    tone = ("The prediction discriminates TEP amplitude, but the resulting trials are not "
            "measurably larger than controls. The mechanism shows without the intervention "
            "effect reaching significance at this sample size.")
elif effect and not established:
    tone = ("PRIME trials show larger TEPs than controls, but the prediction value itself "
            "does not track amplitude within the delivered range. Check the confounds in "
            "section 3 before attributing the difference to brain-state selection.")
else:
    tone = ("Neither claim is established in this session. The prediction does not track TEP "
            "amplitude within the delivered range, and PRIME trials are not measurably "
            "larger than controls.")
print(f"\n{tone}\n")

print("Three things that bound that conclusion:")
print(f"  1. CEILING. Lag-1 autocorrelation of the outcome is {r1:+.3f}. "
      + ("With no serial structure, little is predictable in principle."
         if abs(r1) < .1 else "Some structure exists to exploit."))
if PRED_LOG is not None:
    print(f"  2. TRUNCATION. {100*(1 - len(trig)/len(allp)):.0f}% of scored moments never "
          f"became a pulse, mechanically shrinking the Q1 correlation.")
print(f"  3. POWER. n = {n1} vs {n2}; effects below d = {d_det:.2f} are invisible here.")

print("\nThe decisive next step is section 7: score the predetermined trials retrospectively,")
print("or log a shadow prediction on them in future sessions. Either removes the truncation")
print("and turns Q1 into a properly powered test.")


In [ ]:
# ---------------------------------------------------------------- export
tables = {"q1_correlations": Q1CORR, "q1_by_block": BLKCORR, "q2_comparison": Q2,
          "q2_adjusted": ADJ, "q2_by_block": BLK2, "confounds": CONFOUNDS,
          "autocorrelation": AC, "verdict": VERDICT}
if PRED_LOG is not None:
    tables["range_restriction"] = RR
if len(NEG):
    tables["negative_controls"] = NEG

for name, df in tables.items():
    df.to_csv(OUT_DIR / f"performance_{name}.csv", index=False)

keep = [c for c in ["stage", "trial_in_stage", "condition", "pulse_time", "t_min", "isi",
                    "prev_condition", "kept", "prediction_probability", "prime_attempts",
                    "qc_failures", "tep_amplitude", "tep_amplitude_raw", "tep_zscore",
                    "amp_replay", "r2_free", "outcome"] if c in S.columns]
S[keep].to_csv(OUT_DIR / "performance_trial_table.csv", index=False)

print(f"wrote {len(tables) + 1} files to {OUT_DIR}")
for name in list(tables) + ["trial_table"]:
    print(f"   performance_{name}.csv")


---

### Reading this on a new session

Three cells carry the conclusion: **section 2** (is anything predictable at all), **section 4**
(does the prediction discriminate), **section 8** (the assembled verdict). Section 3 tells you
whether the comparison in section 5 can be believed.

Every number is recomputed from that session's data — nothing here is carried over from the
session it was written against. The bootstrap and permutation seeds are fixed, so results are
reproducible run to run.
